# 第 1 周练习：网页抓取 → 梵语摘要

## 练习目标

把「抓取网页正文 + Chat Completions」串成一条流水线：输入一个 URL，输出**简明梵语（Sanskrit）摘要**。

这是 Day 1 网站摘要项目的变体：模型不只做英文总结，还要跨语言理解并译写为目标语。

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Environment Variables / `.env` | `load_dotenv` + `OPENAI_API_KEY` |
| 网页抓取 | `fetch_website_contents(url)`（来自 `scraper`） |
| system / user prompt | system 定角色与规则，user 放网页正文 |
| Chat Completions | `client.chat.completions.create(...)` |

## 怎么跑

1. 确保项目环境已激活，且 `.env` 里有可用的 `OPENAI_API_KEY`
2. 从上到下依次运行单元格（Shift+Enter）
3. 可改 `url` 指向其他商务/资讯页，观察梵语摘要是否仍守规则（专有名词保留、忽略导航噪音等）


In [1]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从本地 scraper 模块导入抓取函数：给定 URL，返回网页正文文本
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具（本格先导入，后面若要 Markdown 渲染可用）
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI


In [2]:
# ========== 环境变量：加载 .env 并做最基本的密钥体检 ==========

# override=True：若进程里已有同名环境变量，也用 .env 覆盖（与原代码一致，勿改参数）
load_dotenv(override=True)
# 从环境读取 OpenAI API Key；键名必须是 OPENAI_API_KEY（不要改成别的名字）
api_key = os.getenv('OPENAI_API_KEY')

# 检查钥匙：没有 key 就打印英文提示（文案影响排错指引，保持原文）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
else:
    # 找到 key：只做「存在性」确认；更严的格式检查可对照官方 Day1 笔记本
    print("API key found and looks good so far!")


API key found and looks good so far!


In [15]:
# ========== Prompt：系统角色（梵语摘要规则）+ 用户侧任务模板 ==========

# system_prompt：告诉模型「你是谁、怎么做」——字符串内容必须保持英文原文，勿翻译
system_prompt = """
You are a translation and summarization assistant. You will be given text scraped from 
a webpage, which may be in German, French, Italian, or Romansh (Swiss national languages), 
or other European languages. Your job is to:

1. Understand the original text regardless of source language.
2. Produce a clear, plain-Sanskrit summary — as if explaining it to someone with no 
   knowledge of the source language or specialist context.
3. Preserve names, dates, numbers, and proper nouns exactly as they appear in the source.
4. If the scraped content includes irrelevant boilerplate (navigation menus, cookie 
   notices, ads, footers), ignore it and summarize only the substantive content.
5. If the page content is too fragmented or noisy to summarize meaningfully, say so 
   rather than inventing a coherent narrative.

Do not add outside information. Only summarize what is present in the given text.
"""
# user_prompt：用户消息前缀；后面会再拼接 webpage_content（正文）
# f-string 在这里没有插值变量，但保持原写法，避免逻辑漂移
user_prompt = f"""
Summarize the following webpage content in plain Sanskrit. Keep it concise (aim for 
150-250 words unless the source is very short), and organize it with a short heading 
plus 3-5 key points if the content has multiple distinct topics.

Webpage content:

"""


In [12]:
# ========== 抓取：选定 URL，把网页正文拉进变量 ==========

# 目标页：瑞士商务指南（示例 URL，勿改字符串以免改变实验对象）
url = "https://www.andiamo.co.uk/resources/country-business-guides/switzerland/"
# fetch_website_contents：封装了请求 + 清洗；返回可供模型阅读的纯文本
webpage_content = fetch_website_contents(url)


In [16]:
# ========== messages：拼成 OpenAI Chat Completions 期望的角色列表 ==========

# 列表里两条：system 定规则，user = 任务说明 + 抓取正文
messages = [
    {"role": "system", "content": system_prompt},
    # 字符串拼接：user_prompt 前缀 + webpage_content 正文（顺序不要改）
    {"role": "user", "content": user_prompt + webpage_content}
]


In [ ]:
# ========== 调用 API：用密钥建客户端，发 chat.completions，打印回复 ==========

# 用前面读到的 api_key 初始化 OpenAI 客户端（显式传参，不依赖默认环境探测）
client = OpenAI(api_key=api_key)

# chat.completions.create：非流式一次返回完整回复
response = client.chat.completions.create(
    # 模型 id 必须保持 gpt-4o-mini（影响计费与行为，勿改）
    model="gpt-4o-mini",
    # 上面组装好的 system + user
    messages=messages,
    # temperature=0.0：尽量稳定、少随机（翻译/摘要类常用）
    temperature=0.0
)

# choices[0].message.content：取第一条候选的文本内容并打印
print(response.choices[0].message.content)
